# SARIMAX (statsmodels) vs Arima (R)

ptional check: the final ARIMA(0,1,2) model of `01_arima` is re-estimated with
`forecast::Arima` in R, and the two packages are compared in terms of parameters,
standard errors, log-likelihood and ex-post forecasts.

**Requirements:** R with the `forecast` package, and `rpy2` (`pip install rpy2`).
Notebooks 03–05 do not depend on this one and can be run without R.

## Setup

In [1]:
%load_ext rpy2.ipython
from pathlib import Path

import numpy as np
import pandas as pd

from src.config import HORIZONS, N_TEST
from src.evaluation import error_metrics
from src.models import fit_sarimax_with_exog

DATA_DIR = Path("data")
data = pd.read_csv(DATA_DIR / "model_data.csv", index_col="date", parse_dates=True).asfreq("D")
y, X = data["x"], data.drop(columns="x")
arima_py = pd.read_csv(DATA_DIR / "arima_forecasts.csv", index_col="date", parse_dates=True)

res_py = fit_sarimax_with_exog(y.iloc[:-N_TEST], X.iloc[:-N_TEST], order=(0, 1, 2), trend="n")


In [2]:
# plain numpy objects to pass to R
y_r = y.to_numpy(dtype=float)
X_r = X.to_numpy(dtype=float)
xreg_names = X.columns.to_numpy(dtype=str)
horizons_r = np.array(HORIZONS)
n_test = N_TEST

# forecast::Arima

In [3]:
%%R -i y_r -i X_r -i xreg_names -i n_test -o r_coef -o r_stats
library(forecast)
colnames(X_r) <- xreg_names
n_train <- length(y_r) - n_test

fit_r <- Arima(y_r[1:n_train], order = c(0, 1, 2), xreg = X_r[1:n_train, ],
            include.drift = FALSE, method = "ML")
print(fit_r)

r_coef <- data.frame(name = names(coef(fit_r)), coef = unname(coef(fit_r)),
                    se = sqrt(diag(fit_r$var.coef)))
r_stats <- c(fit_r$loglik, fit_r$aic, fit_r$bic)


Series:

y_r[1:n_train]

Regression with ARIMA(0,1,2) errors


Coefficients:


      ma1

...

   AICc=

20109.62

   BIC=

20262.59

Registered S3 method overwritten by 'quantmod':
  method            from
  as.zoo.data.frame zoo 


## Parameters and SE

In [4]:
r = r_coef.set_index("name").rename(index={"ma1": "ma.L1", "ma2": "ma.L2"})
params = res_py.params.drop("sigma2")

comparison = pd.DataFrame({
    "Python": params,
    "R": r["coef"],
    "SE Python": res_py.bse.drop("sigma2"),
    "SE R": r["se"],
}, index=params.index)
comparison["difference"] = comparison["Python"] - comparison["R"]
display(comparison.round(4))


,Python,R,SE Python,SE R,difference
25_dec,-86.9089,-86.6222,17.7062,9.5829,-0.2868
15_aug,-84.1847,-84.3343,15.9342,11.2220,0.1495
31_dec,-57.3350,-56.5987,10.4338,9.6606,-0.7362
24_dec,-27.9713,-28.2063,13.8719,9.5855,0.2350
2_jan,59.5776,61.4371,7.5331,9.9613,-1.8595
27_dec,50.1483,50.9505,11.3281,9.4887,-0.8022
Mon,41.0898,41.1125,1.7603,1.7096,-0.0227
Tue,11.3593,11.3908,1.7064,1.7792,-0.0314
Wed,6.5874,6.6099,1.7494,1.7756,-0.0225
Thu,9.9060,9.9618,1.8661,1.7907,-0.0559


## Likelihood and information criteria

In [5]:
# Python log-likelihood evaluated at R's estimates
at_r = res_py.params.copy()
at_r[r.index] = r["coef"]

fit_stats = pd.DataFrame(
    {"Python": [res_py.llf, res_py.aic, res_py.bic], "R": list(r_stats)},
    index=["log-likelihood", "AIC", "BIC"],
)
display(fit_stats.round(3))
print(f"Python log-likelihood at R's estimates: {res_py.model.loglike(at_r):.3f}")


,Python,R
log-likelihood,-10027.486,-10027.458
AIC,20108.971,20108.916
BIC,20262.646,20262.591


Python log-likelihood at R's estimates: -10027.455


## Ex-post forecast with fixed parameters, origin t - h

In [6]:
%%R -i horizons_r -o r_fc
r_fc <- matrix(NA_real_, n_test, length(horizons_r))
for (j in seq_along(horizons_r)) {
    h <- horizons_r[j]
    for (i in seq_len(n_test)) {
        t <- n_train + i
        origin <- t - h
        fit_t <- Arima(y_r[1:origin], xreg = X_r[1:origin, , drop = FALSE], model = fit_r)
        r_fc[i, j] <- forecast(fit_t, h = h, xreg = X_r[(origin + 1):t, , drop = FALSE])$mean[h]
    }
}


## Forecast comparison

In [7]:
r_fc = pd.DataFrame(r_fc, index=y.index[-N_TEST:], columns=[f"ARIMA_h{h}" for h in HORIZONS])
y_test = y.iloc[-N_TEST:]

rows = []
for h in HORIZONS:
    col = f"ARIMA_h{h}"
    py, rr = error_metrics(y_test, arima_py[col]), error_metrics(y_test, r_fc[col])
    diff = (arima_py[col] - r_fc[col]).abs()
    rows.append({"h": h, "RMSE Python": py["RMSE"], "RMSE R": rr["RMSE"],
                "MAE Python": py["MAE"], "MAE R": rr["MAE"],
                "mean |Python - R|": diff.mean(), "max |Python - R|": diff.max(),
                "day of max": diff.idxmax().date()})
display(pd.DataFrame(rows).round(3))

,h,RMSE Python,RMSE R,MAE Python,MAE R,mean |Python - R|,max |Python - R|,day of max
0,1,20.346,20.336,16.223,16.213,0.053,1.743,2019-01-02
1,3,20.661,20.646,16.398,16.389,0.060,1.787,2019-01-02
2,7,21.000,20.982,16.747,16.732,0.070,1.836,2019-01-02
3,14,21.644,21.622,17.229,17.217,0.087,1.842,2019-01-02
4,30,22.514,22.497,18.010,17.993,0.114,1.873,2019-01-02
